# Lab 7: AI Vision and Inference

Run each cell with **Shift + Enter**, from the top. Pictures appear right under the cell. Do **not** run `pip install` in this lab. If something is missing, call your instructor.

The live camera detection (Part D) is done in the **terminal**, as described in the manual.

In [ ]:
from IPython.display import Image, display

def show(*names, width=360):
    """Show picture files under the cell."""
    for n in names:
        print(n)
        display(Image(filename=n, width=width))

## Part A, Step 2: Check PyTorch and YOLO
If `CUDA available` is `False` or the PyTorch version ends with `+cpu`, stop and call your instructor.

In [ ]:
import torch
import ultralytics

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
print("Ultralytics version:", ultralytics.__version__)

## Part A, Step 3: Get the Sample Pictures and Models
The YOLO package comes with two sample pictures. This copies them into the folder of this notebook. It also copies the model files that `setup.sh` downloaded, so nothing is downloaded during the lab.

In [ ]:
import glob, os, shutil
import ultralytics

assets = os.path.join(os.path.dirname(ultralytics.__file__), "assets")
for f in glob.glob(assets + "/*.jpg"):
    shutil.copy(f, ".")

# Model files downloaded by setup.sh
for f in glob.glob(os.path.expanduser("~/jetson-labs/models/*.pt")):
    shutil.copy(f, ".")

print(sorted(glob.glob("*.jpg")))
print(sorted(glob.glob("*.pt")))
show("bus.jpg", "zidane.jpg")

## Part B, Step 1: Classify an Image
The classifier gives **one list** of guesses for the whole picture. The first run takes longer because the model loads and the GPU prepares. The model file was copied in the step above, so no internet is needed.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
cls_model = YOLO("yolo26n-cls.pt")

def classify(image):
    result = cls_model(image, device=device, verbose=False)[0]
    print(f"Image: {image}   Image size: {result.orig_shape}")
    print(f"Model knows {len(result.names)} classes")
    print("Top 5 guesses:")
    for rank, i in enumerate(result.probs.top5, start=1):
        print(f"  {rank}. {result.names[i]:<20} {float(result.probs.data[i]) * 100:5.1f} %")
    print(f"Inference time: {result.speed['inference']:.1f} ms")

classify("bus.jpg")
print()
classify("zidane.jpg")

## Part B, Step 2: Test Your Own Pictures
Use the picture from Lab 6 (`snapshot.jpg`, saved by the Lab 6 notebook). Or put any picture in this folder and change the name.

In [ ]:
import os

snap = "../lab6/snapshot.jpg"      # the Lab 6 notebook saves its pictures in ~/jetson-labs/lab6
if os.path.exists(snap):
    classify(snap)
    show(snap)
else:
    print("The picture snapshot.jpg does not exist yet.")
    print("It is the photo your USB camera takes in the Lab 6 notebook, cell Part B, Step 3.")
    print("Go back to Lab 6, run that cell, then come back here.")
    print("Or skip it: copy any .jpg picture into this folder and use its name instead.")

## Part C, Step 1: Detect Objects
The detector finds every object it knows, draws a box around it and gives its class and confidence. The second value, `conf`, is the confidence limit.

In [ ]:
import cv2

det_model = YOLO("yolo26n.pt")

def detect(image, conf=0.25, save_as="detected.jpg"):
    result = det_model(image, conf=conf, device=device, verbose=False)[0]
    print(f"Image: {image}   Confidence limit: {conf}")
    print(f"Found {len(result.boxes)} objects:")
    for box in result.boxes:
        name = result.names[int(box.cls)]
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        print(f"  {name:<12} {float(box.conf):.2f}   box: ({x1}, {y1}) to ({x2}, {y2})")
    cv2.imwrite(save_as, result.plot())
    return result

detect("bus.jpg")
show("detected.jpg")

In [ ]:
detect("zidane.jpg")
show("detected.jpg")

## Part C, Step 2: Change the Confidence Limit
This runs the detector on `bus.jpg` with four different limits. It saves one picture for each, so you can compare the lowest and the highest.

In [ ]:
for limit in [0.10, 0.25, 0.50, 0.80]:
    detect("bus.jpg", conf=limit, save_as=f"detected_{limit}.jpg")
    print()

show("detected_0.1.jpg", "detected_0.8.jpg")

## Part C, Step 3: Look at the Box Numbers
Work out the width and height of the **bus** box by hand first (`x2 - x1` and `y2 - y1`). Then run this cell to check your answer.

In [ ]:
result = det_model("bus.jpg", conf=0.25, device=device, verbose=False)[0]
for box in result.boxes:
    if result.names[int(box.cls)] == "bus":
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        print("Bus box:", (x1, y1), "to", (x2, y2))
        print("Width :", x2 - x1)
        print("Height:", y2 - y1)
        print("Touches the left edge:", x1 <= 2)

## Part E, Step 1: Measure the Speed Baseline
This measures only the time the model needs for one picture, for the nano and the small model. It runs 5 warm-up passes and then 30 timed runs. Close the live window and your other notebooks first, then run the cell **twice** and record the second run. Write the power mode in the manual table.

In [ ]:
import time
import torch
from ultralytics import YOLO

image = "bus.jpg"
device = 0 if torch.cuda.is_available() else "cpu"
names = ["yolo26n.pt", "yolo26s.pt"]
runs = 30

for name in names:
    model = YOLO(name)
    for _ in range(5):                       # warm-up
        model(image, imgsz=640, device=device, verbose=False)

    times = []
    for _ in range(runs):
        r = model(image, imgsz=640, device=device, verbose=False)[0]
        times.append(r.speed["inference"])   # milliseconds, model only

    avg = sum(times) / len(times)
    print(f"{name:<12} average inference: {avg:6.1f} ms   ({1000 / avg:5.1f} FPS)")

### Try It Yourself
- Change `imgsz=640` to `320` in the cell above. How does the time change?